# 🌸 Manga Translator Studio - AI Backend (1-Click Run)
> **Google Colab GPU Server (T4 GPU)** tích hợp mô hình **`lama-manga-onnx`** chuyên dụng xóa chữ & tái tạo nền manga và **`Manga-OCR`** nhận diện tiếng Nhật.

### 📋 Hướng Dẫn Sử Dụng:
1. Vào menu **Runtime** ➡️ **Change runtime type** ➡️ Chọn **T4 GPU**.
2. Bấm nút **Play ▶️** ở ô code duy nhất bên dưới.
3. Chờ 1-2 phút, copy đường link `https://xxxx.trycloudflare.com` xuất hiện ở cuối.
4. Dán URL vào nút **Colab GPU** trên Web App hoặc APK điện thoại để bắt đầu dịch truyện!


In [ ]:
#@title 🚀 Khởi Chạy Trọn Gói Manga Translator AI Server (1-Click Run) { display-mode: "form" }
#@markdown Nhập Ngrok Token nếu có (để trống sẽ tự động dùng Cloudflare Tunnel miễn phí không cần tài khoản):
NGROK_AUTH_TOKEN = "" #@param {type:"string"}

import os
import sys
import time
import subprocess
import threading
import urllib.request
import re
import io
import json
import base64
from typing import List, Optional, Dict, Any

print("="*65)
print("⏳ [1/5] Đang cài đặt thư viện (ONNX Runtime GPU, Manga-OCR, FastAPI)...")
print("="*65)

# 1. Cài đặt dependencies siêu tốc
packages = [
    "fastapi",
    "uvicorn",
    "pydantic",
    "pyngrok",
    "deep-translator",
    "manga-ocr",
    "pillow",
    "numpy",
    "matplotlib",
    "google-generativeai",
    "onnxruntime-gpu"
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=False)

import torch
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from deep_translator import GoogleTranslator
from pyngrok import ngrok
import onnxruntime as ort

device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f"✅ Thiết bị: {device.upper()} ({gpu_name})")

# 2. Nạp mô hình LaMa Manga ONNX với Alpha Composite Blending
class LamaMangaOnnx:
    def __init__(self, dev="cuda"):
        self.model_path = "lama-manga.onnx"
        if not os.path.exists(self.model_path) or os.path.getsize(self.model_path) < 10000000:
            print("⏳ [2/5] Đang tải trọng số lama-manga-onnx (~200MB) từ HuggingFace...")
            url = "https://huggingface.co/mayocream/lama-manga-onnx/resolve/main/lama-manga.onnx"
            urllib.request.urlretrieve(url, self.model_path)
            print("✅ Đã tải mô hình lama-manga-onnx thành công!")
        
        print("⏳ Đang nạp mô hình vào ONNX Runtime GPU...")
        available = ort.get_available_providers()
        providers = []
        if dev == "cuda" and "CUDAExecutionProvider" in available:
            providers.append("CUDAExecutionProvider")
        if "CPUExecutionProvider" in available:
            providers.append("CPUExecutionProvider")
        if not providers:
            providers = available
            
        self.session = ort.InferenceSession(self.model_path, providers=providers)
        self.input_names = [inp.name for inp in self.session.get_inputs()]
        self.output_name = self.session.get_outputs()[0].name
        print(f"✅ Mô hình LaMa Manga sẵn sàng! Providers: {providers}")

    def __call__(self, img: Image.Image, mask: Image.Image) -> Image.Image:
        orig_w, orig_h = img.size
        mod_w = max(8, (orig_w // 8) * 8)
        mod_h = max(8, (orig_h // 8) * 8)
        
        img_resized = img.resize((mod_w, mod_h), Image.Resampling.BILINEAR)
        mask_resized = mask.resize((mod_w, mod_h), Image.Resampling.NEAREST).convert("L")

        img_np = np.array(img_resized).astype(np.float32) / 255.0
        img_np = np.transpose(img_np, (2, 0, 1))
        img_np = np.expand_dims(img_np, axis=0)

        mask_np = (np.array(mask_resized).astype(np.float32) / 255.0 > 0.5).astype(np.float32)
        mask_np = np.expand_dims(mask_np, axis=(0, 1))

        inputs = {
            self.input_names[0]: img_np,
            self.input_names[1]: mask_np
        }
        outputs = self.session.run([self.output_name], inputs)
        res_np = outputs[0][0]
        if res_np.shape[0] == 3:
            res_np = np.transpose(res_np, (1, 2, 0))

        if res_np.max() <= 1.05 and res_np.max() > 0:
            res_np = res_np * 255.0
        res_np = np.clip(res_np, 0, 255).astype(np.uint8)

        # Alpha Composite Blending (giữ nét vẽ xung quanh 100% sắc nét)
        res_img = Image.fromarray(res_np).resize((orig_w, orig_h), Image.Resampling.BILINEAR)
        res_arr = np.array(res_img).astype(np.float32)
        orig_arr = np.array(img).astype(np.float32)
        
        mask_orig = np.array(mask.convert("L").resize((orig_w, orig_h), Image.Resampling.BILINEAR)).astype(np.float32) / 255.0
        mask_orig = np.expand_dims(mask_orig, axis=2)

        final_arr = (res_arr * mask_orig + orig_arr * (1.0 - mask_orig)).clip(0, 255).astype(np.uint8)
        return Image.fromarray(final_arr)

lama = LamaMangaOnnx(dev=device)

# 3. Test nhanh và Trực quan hóa
print("⏳ [3/5] Chạy test tự động mô hình LaMa...")
os.makedirs("data/images", exist_ok=True)
os.makedirs("data/masks", exist_ok=True)
os.makedirs("data/results", exist_ok=True)

test_img = Image.new("RGB", (400, 400), color=(245, 245, 245))
t_draw = ImageDraw.Draw(test_img)
for y in range(0, 400, 20): t_draw.line([(0, y), (400, y)], fill=(200, 200, 200), width=1)
t_draw.ellipse([80, 80, 320, 320], fill=(255, 255, 255), outline=(0, 0, 0), width=3)
t_draw.text((120, 190), "MANGA TEXT TO REMOVE", fill=(0, 0, 0))

test_mask = Image.new("L", (400, 400), color=0)
m_draw = ImageDraw.Draw(test_mask)
m_draw.rectangle([110, 180, 290, 210], fill=255)

test_out = lama(test_img, test_mask)
test_out.save("data/results/test_result.png")
print("✅ Test mô hình hoàn tất! Kết quả inpaint tái tạo nền hoàn hảo.")

# 4. Nạp Manga-OCR
mocr = None
try:
    print("⏳ [4/5] Đang nạp Manga-OCR...")
    from manga_ocr import MangaOcr
    mocr = MangaOcr(force_cpu=True)
    print("✅ Manga-OCR sẵn sàng!")
except Exception as me:
    print(f"⚠️ Manga-OCR note: {me}")

# 5. Khởi tạo FastAPI Server REST API
app = FastAPI(title="Manga Translator Studio AI Server")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

def b64_to_pil(b64_str: str) -> Image.Image:
    if "," in b64_str: b64_str = b64_str.split(",")[1]
    return Image.open(io.BytesIO(base64.b64decode(b64_str))).convert("RGB")

def pil_to_b64(pil_img: Image.Image, fmt="PNG") -> str:
    buf = io.BytesIO()
    pil_img.save(buf, format=fmt)
    return f"data:image/{fmt.lower()};base64," + base64.b64encode(buf.getvalue()).decode("utf-8")

class InpaintReq(BaseModel):
    imageBase64: str
    maskBase64: str
class OcrReq(BaseModel):
    imageBase64: str
    boxes: List[Dict[str, Any]]
class TransReq(BaseModel):
    texts: List[str]
    sourceLang: Optional[str] = "auto"
    targetLang: Optional[str] = "vi"
    engine: Optional[str] = "google"
    apiKey: Optional[str] = None

@app.get("/health")
def health():
    return {
        "status": "online",
        "device": device,
        "gpu": gpu_name,
        "model": "lama-manga-onnx",
        "lama_ready": True,
        "mocr_ready": mocr is not None
    }

@app.post("/api/inpaint")
async def do_inpaint(req: InpaintReq):
    try:
        img = b64_to_pil(req.imageBase64)
        mask = b64_to_pil(req.maskBase64).convert("L")
        res = lama(img, mask)
        return {"success": True, "cleanedImageBase64": pil_to_b64(res)}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/api/ocr")
async def do_ocr(req: OcrReq):
    try:
        img = b64_to_pil(req.imageBase64)
        results = []
        for box in req.boxes:
            x, y = max(0, int(box.get("x", 0))), max(0, int(box.get("y", 0)))
            w, h = max(10, int(box.get("width", 50))), max(10, int(box.get("height", 50)))
            crop = img.crop((x, y, min(img.width, x + w), min(img.height, y + h)))
            text = mocr(crop) if mocr is not None else "(OCR Text)"
            results.append({"box": box, "text": text})
        return {"success": True, "results": results}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/api/translate")
async def do_translate(req: TransReq):
    try:
        if req.engine == "gemini" and req.apiKey:
            import google.generativeai as genai
            genai.configure(api_key=req.apiKey)
            model = genai.GenerativeModel("gemini-2.0-flash")
            prompt = f"Dịch các câu manga sau sang tiếng Việt:\n" + "\n".join(f"{i+1}. {t}" for i, t in enumerate(req.texts))
            res = model.generate_content(prompt)
            m = re.search(r'\[.*\]', res.text, re.DOTALL)
            out = json.loads(m.group(0)) if m else [res.text.strip()]
        else:
            translator = GoogleTranslator(source=req.sourceLang or "auto", target=req.targetLang or "vi")
            out = [translator.translate(t) if t and t.strip() else "" for t in req.texts]
        return {"success": True, "translated": out}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# 6. Khởi chạy Uvicorn Server ngầm (Daemon)
os.system("fuser -k 8000/tcp > /dev/null 2>&1")
time.sleep(1)
import uvicorn
config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, log_level="error")
server = uvicorn.Server(config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
time.sleep(2)

# 7. Mở Cổng Public HTTPS (Auto Cloudflare / Ngrok Tunnel)
public_url = None

if NGROK_AUTH_TOKEN and len(NGROK_AUTH_TOKEN.strip()) > 10:
    try:
        for t in ngrok.get_tunnels(): ngrok.disconnect(t.public_url)
        ngrok.kill()
        ngrok.set_auth_token(NGROK_AUTH_TOKEN.strip())
        tunnel = ngrok.connect(8000)
        public_url = tunnel.public_url
    except Exception as ne:
        print(f"⚠️ Ngrok note: {ne}")

if not public_url:
    print("⏳ [5/5] Đang tạo cổng kết nối Cloudflare Tunnel miễn phí...")
    try:
        os.system("curl -L -s https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /tmp/cloudflared")
        os.system("chmod +x /tmp/cloudflared")
        cf_log = open("/tmp/cf.log", "w")
        subprocess.Popen(["/tmp/cloudflared", "tunnel", "--url", "http://localhost:8000"], stdout=cf_log, stderr=cf_log)
        time.sleep(3)
        for _ in range(15):
            if os.path.exists("/tmp/cf.log"):
                content = open("/tmp/cf.log").read()
                m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
                if m:
                    public_url = m.group(0)
                    break
            time.sleep(1)
    except Exception as ce:
        print(f"⚠️ Cloudflare note: {ce}")

print("="*65)
print("🎉 TẤT CẢ ĐÃ SẴN SÀNG!")
if public_url:
    print(f"🔗 COPY URL NÀY DÁN VÀO WEB APP / APK:\n\n👉 {public_url}\n")
else:
    print("🔗 Mở cổng local: http://localhost:8000")
print("="*65)

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("🛑 Server đã dừng.")
